# 🎯 [PRC07_M05] 공급망 붕괴 위험도(Supply Chain Risk Score) 및 조기경보 산출 엔진 개발 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M05 |
| **미션 주제** | 공급망 붕괴 위험도(Supply Chain Risk Score) 및 조기경보 산출 엔진 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 05. 공급망 붕괴 위험도(Supply Chain Risk Score) 및 조기경보 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

공급망 붕괴 위험도 및 조기경보 산출 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **공급망 붕괴 위험도(Supply Chain Risk Score)와 월별 지연 추이 기반 조기경보를 산출하는 엔진을 만들고 입력-분석-검토-결과 4개 화면으로 구성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **공급선 수, 해외 조달, 리드타임, 납기 지연률, 단종 상태 등 12개 지표로 로지스틱 회귀 위험도(0~100점)를 학습하고 별도 가상 부품군에서 AUC와 등급별 실제 단절률 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/prc07_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""12개 공급망 지표로 부품별 공급망 붕괴 위험도(0~100점)를 로지스틱 회귀로 산출하고 월별 납기 지연 추이의 EWMA로 조기경보를 내는 엔진을 만든 뒤, 입력-분석-검토-결과 4개 화면(텍스트 패널과 그래프)을 구성해 별도 가상 부품군에서 성능을 검증한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    # [TODO : 학생 실습 5] 공급망 붕괴 위험도(Supply Chain Risk Score) 및 조기경보 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sig(x):
    # [TODO : 학생 실습 5] 공급망 붕괴 위험도(Supply Chain Risk Score) 및 조기경보 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 공급망 붕괴 위험도(Supply Chain Risk Score) 공통 함수: 로지스틱 회귀 + 월별 지연 추이 조기경보 ----
RISK_FEATS = ["n_suppliers", "overseas", "lead_time_days", "delay_rate", "avg_delay_days", "eol_ord", "ltb_short", "price_change_pct",
              "crit_ord", "cover_gap_days", "defect_rate", "export_license"]
FEAT_KO = {"n_suppliers": "공급선 수", "overseas": "해외 조달", "lead_time_days": "리드타임", "delay_rate": "납기 지연률", "avg_delay_days": "평균 지연일",
           "eol_ord": "단종 상태", "ltb_short": "최종구매 임박", "price_change_pct": "단가 변동", "crit_ord": "체계 중요도",
           "cover_gap_days": "재고-리드타임 격차", "defect_rate": "입고 불량률", "export_license": "수출 허가 필요"}
WARN_CUT = {"경보": 60, "주의": 35}          # 위험도 점수(0~100) 등급 기준


def risk_frame(parts):
    d = parts.copy()
    d["delay_rate"] = d["late_orders"] / d["orders_12m"].clip(lower=1)
    d["eol_ord"] = d["eol_status"].map({"정상": 0, "단종예고": 1, "단종": 2})
    d["ltb_short"] = (d["ltb_months"].fillna(99) <= 6).astype(int)
    d["crit_ord"] = d["criticality"].map({"C": 0, "B": 1, "A": 2})
    d["cover_gap_days"] = d["stock_qty"] / d["annual_demand"].clip(lower=1) * 365 - d["lead_time_days"]
    for c in ("overseas", "export_license"):
        d[c] = d[c].astype(int)
    return d


def fit_risk(parts):
    d = risk_frame(parts)
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500)).fit(d[RISK_FEATS], d["disrupted_next12m"].astype(int))


def risk_score(model, parts):
    """붕괴 위험도 0~100점 = 다음 12개월 내 공급 단절 확률 x 100."""
    return model.predict_proba(risk_frame(parts)[RISK_FEATS])[:, 1] * 100


def drivers(model, parts, k=3):
    """부품별 위험 상승 요인 상위 k개: (표준화 값 x 회귀 계수)가 큰 순서."""
    sc, lr = model[0], model[-1]
    z = sc.transform(risk_frame(parts)[RISK_FEATS]) * lr.coef_[0]
    return [[FEAT_KO[RISK_FEATS[j]] for j in np.argsort(-row)[:k] if row[j] > 0] for row in z]


def monthly_delay(parts, seed=5):
    """[교육용가상자료] 부품별 최근 12개월 평균 납기 지연일. 단절될 부품은 단절 3~6개월 전부터 지연이 서서히 늘어난다."""
    rng = np.random.default_rng(seed)
    out, onset = [], []
    for _, p in parts.iterrows():
        base = 3 + 0.15 * p["avg_delay_days"]
        s = base + rng.normal(0, 1.2, 12)
        o = -1
        if p["disrupted_next12m"] and rng.random() < 0.7:
            o = int(rng.integers(5, 9))                       # 12개월 구간 내 상승 시작 월
            s[o:] += np.linspace(1.5, 12, 12 - o) + rng.normal(0, 1.0, 12 - o)
        out.append(s); onset.append(o)
    return np.array(out), np.array(onset)


def ewma_alarm(series, base_n=5, lam=0.3, k=3.0):
    """처음 base_n개월을 기준선으로 삼아 EWMA가 기준선 + k x 표준편차를 넘는 첫 달(없으면 -1)."""
    mu, sd = series[:base_n].mean(), series[:base_n].std() + 0.5
    e = mu
    for t in range(base_n, len(series)):
        e = lam * series[t] + (1 - lam) * e
        if e > mu + k * sd * np.sqrt(lam / (2 - lam)):
            return t
    return -1

train, _ = make_parts(600, seed=11)           # 학습용 가상 부품군
test, _ = make_parts(200, seed=7)             # 검증용 별도 가상 부품군
model = fit_risk(train)
test = test.assign(score=risk_score(model, test), why=drivers(model, test))
y = test["disrupted_next12m"].astype(int)
test["level"] = np.where(test["score"] >= WARN_CUT["경보"], "경보", np.where(test["score"] >= WARN_CUT["주의"], "주의", "정상"))

# ===== 화면 1. 입력: 입력 요약과 검증 =====
miss = int(test[["lead_time_days", "n_suppliers", "late_orders"]].isna().sum().sum())
print("[화면 1. 입력] [교육용가상자료]")
print(f"  검증용 부품 {len(test)}종 / 입력 지표 {len(RISK_FEATS)}개 / 필수값 결측 {miss}건 / 학습 부품 {len(train)}종")
print(f"  해외 조달 {int(test['overseas'].sum())}종, 단종예고·단종 {int((test['eol_status'] != '정상').sum())}종, 단일 공급선 {int((test['n_suppliers'] == 1).sum())}종")

# ===== 화면 2. 분석: 위험도 분포와 성능 =====
auc = roc_auc_score(y, test["score"])
print("\n[화면 2. 분석] 위험도 점수 분포와 모델 성능")
print(f"  ROC-AUC {auc:.3f}, Brier {brier_score_loss(y, test['score'] / 100):.3f} (단절 실제 비율 {y.mean():.0%})")
print(test.groupby("level").agg(부품수=("part_no", "size"), 평균점수=("score", "mean"), 실제단절률=("disrupted_next12m", "mean")).reindex(["경보", "주의", "정상"]).round(2).to_string())
coef = pd.Series(model[-1].coef_[0], index=[FEAT_KO[f] for f in RISK_FEATS]).sort_values(ascending=False)
print("  위험 상승 요인(회귀 계수 상위 4):", ", ".join(f"{k} {v:+.2f}" for k, v in coef.head(4).items()))

# ===== 조기경보: 월별 지연 추이 =====
md, onset = monthly_delay(test)
alarm = np.array([ewma_alarm(s) for s in md])
dis = onset >= 0
det = (alarm[dis] >= 0).mean()
delay = (alarm[dis & (alarm >= 0)] - onset[dis & (alarm >= 0)]).mean()
fa = (alarm[~dis] >= 0).mean()
print(f"\n[조기경보] 지연 상승형 단절 부품 {int(dis.sum())}종 중 {det:.0%} 감지, 상승 시작 후 평균 {delay:.1f}개월 만에 경보, 정상 부품 오경보율 {fa:.1%}")

# ===== 화면 3. 검토: 경보 부품을 담당자가 확인 =====
rev = test[test["level"] == "경보"].sort_values("score", ascending=False)
print(f"\n[화면 3. 검토] 경보 {len(rev)}종 중 상위 5종 (담당자가 근거를 확인하고 조치 여부를 결정)")
for _, r in rev.head(5).iterrows():
    print(f"  {r['part_no']} {r['category']} {r['score']:.0f}점 | 요인: {', '.join(r['why']) or '-'} | 단종 {r['eol_status']}, 공급선 {r['n_suppliers']}개")

# ===== 화면 4. 결과: 요약과 내보내기 =====
top = rev.head(10)
print("\n[화면 4. 결과]")
print(f"  경보 {len(rev)}종, 주의 {int((test['level'] == '주의').sum())}종, 정상 {int((test['level'] == '정상').sum())}종")
print(f"  경보 등급의 실제 단절 {int(rev['disrupted_next12m'].sum())}종({rev['disrupted_next12m'].mean():.0%}) vs 정상 등급 {test[test['level'] == '정상']['disrupted_next12m'].mean():.0%}")
print("  내보내기 항목: 부품번호, 점수, 등급, 요인, 조기경보 월, 검토 상태(미검토)")
print("화면 4종(입력-분석-검토-결과) 구현: 4/4 (목표 4종: 달성)")
print("※ 점수는 우선순위를 정하는 참고값이며, 실제 단절 여부와 조치는 담당자가 확인한다.")

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0, 0].bar(["overseas", "single src", "EOL"], [test["overseas"].sum(), (test["n_suppliers"] == 1).sum(), (test["eol_status"] != "정상").sum()]); ax[0, 0].set_title("Screen 1: input summary (count)")
ax[0, 1].hist([test[y == 0]["score"], test[y == 1]["score"]], bins=15, label=["no disruption", "disrupted"], stacked=True); ax[0, 1].legend(); ax[0, 1].set_title("Screen 2: risk score distribution")
i = int(np.where(dis & (alarm >= 0))[0][0]); ax[1, 0].plot(md[i], marker="o"); ax[1, 0].axvline(alarm[i], color="r", ls="--"); ax[1, 0].set_title("Screen 3: early warning (delay days, red = alarm)")
ax[1, 1].barh(top["part_no"], top["score"], color="tab:red"); ax[1, 1].invert_yaxis(); ax[1, 1].set_title("Screen 4: top-10 alert parts")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[화면 1. 입력] [교육용가상자료]
  검증용 부품 200종 / 입력 지표 12개 / 필수값 결측 0건 / 학습 부품 600종
  해외 조달 159종, 단종예고·단종 60종, 단일 공급선 89종

[화면 2. 분석] 위험도 점수 분포와 모델 성능
  ROC-AUC 0.693, Brier 0.222 (단절 실제 비율 40%)
       부품수   평균점수  실제단절률
level                   
경보      51  76.75   0.59
주의      59  47.63   0.42
정상      90  22.39   0.27
  위험 상승 요인(회귀 계수 상위 4): 납기 지연률 +0.58, 리드타임 +0.50, 단종 상태 +0.48, 수출 허가 필요 +0.12

[조기경보] 지연 상승형 단절 부품 49종 중 100% 감지, 상승 시작 후 평균 1.9개월 만에 경보, 정상 부품 오경보율 2.0%

[화면 3. 검토] 경보 51종 중 상위 5종 (담당자가 근거를 확인하고 조치 여부를 결정)
  P-0066 커넥터 96점 | 요인: 단종 상태, 납기 지연률, 리드타임 | 단종 단종, 공급선 1개
  P-0080 제어보드 96점 | 요인: 단종 상태, 납기 지연률, 리드타임 | 단종 단종, 공급선 1개
  P-0036 베어링 94점 | 요인: 단종 상태, 납기 지연률, 공급선 수 | 단종 단종, 공급선 1개
  P-0003 유압밸브 94점 | 요인: 단종 상태, 납기 지연률, 리드타임 | 단종 단종, 공급선 1개
  P-0140 베어링 94점 | 요인: 리드타임, 납기 지연률, 공급선 수 | 단종 정상, 공급선 1개

[화면 4. 결과]
  경보 51종, 주의 59종, 정상 90종
  경보 등급의 실제 단절 30종(59%) vs 정상 등급 27%
  내보내기 항목: 부품번호, 점수, 등급, 요인, 조기경보 월, 검토 상태(미검토)
화면 4종(입력-분석-검토-결과) 구현: 4/4 (목표 4종: 달성)
※ 점수는 우선순위를 정하는 참고값이며, 실제 단절 여부와 조치는 담당자가 확인한다.
```
</details>
